# v211 paper-aligned post-2023 h2 fold48 all-model Colab runner

이 노트북은 **Colab 링크를 열고 셀을 순서대로 실행하면** 아래 CLI와 같은 파이프라인이 실행되도록 만든 self-contained 실행본입니다. 기본 데이터는 GitHub의 `data/df_0508.csv`를 자동으로 내려받고, 다운로드가 실패할 때만 수동 업로드로 fallback합니다.

```bash
python scripts/run_main_pipeline.py \
  --data data/df_0508.csv \
  --fixed-start 2011-01-10 \
  --origin-start 2023-01-01 \
  --horizon 2 \
  --step-size 1 \
  --models all \
  --max-origins 48 \
  --out-dir v211_paper_h2_fold48_outputs
```

## 사용 순서
1. Colab 런타임을 GPU로 설정합니다.
2. 패키지 설치 셀을 실행합니다.
3. 데이터 준비 셀을 실행합니다. 기본적으로 `data/df_0508.csv`가 자동 다운로드됩니다.
4. 내장 파이프라인 코드 셀을 실행합니다.
5. 실행 설정/검증 셀을 확인한 뒤 마지막 실행 셀을 실행합니다.

노트북 생성 시점에는 어떤 셀도 실행하지 않았습니다.


In [ ]:
# 0. Install runtime packages in Colab.
# This cell installs package dependencies only. It does not run the forecasting pipeline.
import importlib.util
import subprocess
import sys


def pip_install(packages: list[str]) -> None:
    print("INSTALL:", " ".join(packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


# Colab usually already provides a CUDA-compatible torch build. Install it only if absent.
if importlib.util.find_spec("torch") is None:
    pip_install(["torch", "torchvision", "torchaudio"])

pip_install([
    "neuralforecast>=2.0.0",
    "utilsforecast>=0.2.0",
    "chronos-forecasting>=2.0",
    "pandas[pyarrow]>=2.0",
    "numpy>=1.24",
    "scipy>=1.10",
    "matplotlib>=3.7",
    "tqdm>=4.66",
    "pytorch-lightning>=2.2",
    "accelerate>=0.30",
    "transformers>=4.40",
    "safetensors>=0.4",
    "huggingface_hub>=0.24",
])

print("Dependency installation finished. If Colab asks to restart the runtime, restart and rerun from this cell.")


In [ ]:
# 1. Prepare data/df_0508.csv. Auto-download first; manual upload only if needed.
from pathlib import Path
from urllib.request import urlretrieve

DATA_PATH = Path("data/df_0508.csv")
DATA_URL = "https://raw.githubusercontent.com/Jaeho777/newoil/main/data/df_0508.csv"
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if DATA_PATH.exists() and DATA_PATH.stat().st_size > 0:
    print(f"Using existing {DATA_PATH} ({DATA_PATH.stat().st_size:,} bytes)")
else:
    try:
        print(f"Downloading {DATA_URL}")
        urlretrieve(DATA_URL, DATA_PATH)
        print(f"Saved -> {DATA_PATH} ({DATA_PATH.stat().st_size:,} bytes)")
    except Exception as download_error:
        print("Automatic download failed. Please upload df_0508.csv manually.")
        print(repr(download_error))
        try:
            from google.colab import files
        except ImportError as exc:
            raise RuntimeError("Manual upload fallback is designed for Google Colab.") from exc

        uploaded = files.upload()
        csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
        if len(csv_files) != 1:
            raise ValueError(f"Upload exactly 1 CSV file. Detected CSV files: {csv_files}")
        source_name = csv_files[0]
        DATA_PATH.write_bytes(uploaded[source_name])
        print(f"Saved {source_name} -> {DATA_PATH}")

if DATA_PATH.stat().st_size <= 0:
    raise RuntimeError(f"Prepared CSV is empty: {DATA_PATH}")
print(f"Ready: {DATA_PATH} ({DATA_PATH.stat().st_size:,} bytes)")


## Embedded pipeline code

아래 셀에는 실행에 필요한 파이프라인 코드가 모두 들어 있습니다. 별도 로컬 `.py` 파일이나 저장소 업로드가 필요 없습니다.


In [ ]:
from __future__ import annotations


# ===== Embedded RAG4CTS/config.py =====
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Sequence

import pandas as pd


REPO_ROOT = Path.cwd().resolve()
DEFAULT_OUT_DIR = REPO_ROOT / "v211_paper_h2_fold48_outputs"


@dataclass(frozen=True)
class V210Config:
    """Configuration for the modular v210 NeuralForecast/Chronos workflow.

    Keep this small and explicit: defaults are tuned for comparable
    TimeXer/iTransformer/GRU parameter scale and one-cycle loss diagnostics.
    """

    fixed_start: pd.Timestamp = pd.Timestamp("2011-01-03")
    origin_start: pd.Timestamp = pd.Timestamp("2023-01-01")
    panel_name: str = "requested_post2023_all_step1"
    targets: tuple[str, ...] = (
        "Com_BrentCrudeOil",
        "Com_CrudeOil",
        "Com_DubaiOil",
        "Com_OmanOil",
    )
    horizon: int = 2
    val_size: int = 52
    test_size: int = 2
    step_size: int = 1
    learning_rate: float = 2e-4
    scheduler_kwargs: dict[str, float | int | str] = field(
        default_factory=lambda: {"mode": "min", "factor": 0.5, "patience": 8, "threshold": 0.001}
    )
    scaler_type: str = "identity"
    input_size: int = 64
    batch_size: int = 32
    valid_batch_size: int = 32
    windows_batch_size: int = 128
    inference_windows_batch_size: int = 1024
    max_steps: int = 300
    val_check_steps: int = 50
    early_stop_patience_steps: int = -1
    seed: int = 1
    proposed_input_weeks: int = 16
    proposed_k_grid: tuple[int, ...] = (1, 2, 4, 8, 16)
    proposed_temperature: float = 1.0
    proposed_motif_weight: float = 0.85
    proposed_use_future_covariate: bool = True
    chronos_model_id: str = "amazon/chronos-2"
    quantile_levels: tuple[float, ...] = (0.1, 0.5, 0.9)

    # Local full-run runtime. These defaults intentionally use every visible local GPU.
    # NeuralForecast/PyTorch Lightning: devices=-1 means all available devices.
    # Chronos: use one full model per worker GPU; Chronos-2 layer sharding can produce cross-device tensors.
    require_gpu_for_full_run: bool = True
    lightning_accelerator: str = "gpu"
    lightning_devices: int | str | list[int] = -1
    lightning_strategy: str | None = "auto"
    chronos_device_map: str = "cuda"
    chronos_parallel_devices: int | str | list[int] = -1
    chronos_dtype: str | None = "auto"

    data_path: Path = Path("data/df_0508.csv")
    out_dir: Path = DEFAULT_OUT_DIR

    @property
    def table_dir(self) -> Path:
        return self.out_dir / "tables"

    @property
    def graph_dir(self) -> Path:
        return self.out_dir / "graphs"

    @property
    def run_log_root(self) -> Path:
        return self.out_dir / "runs"

    def with_paths(self, data_path: str | Path | None = None, out_dir: str | Path | None = None) -> "V210Config":
        from dataclasses import replace

        updates = {}
        if data_path is not None:
            updates["data_path"] = Path(data_path)
        if out_dir is not None:
            out = Path(out_dir)
            updates["out_dir"] = out
        return replace(self, **updates)


    def with_local_gpu_runtime(
        self,
        *,
        require_gpu: bool = True,
        lightning_devices: int | str | list[int] = -1,
        lightning_strategy: str | None = "auto",
        chronos_device_map: str = "cuda",
    ) -> "V210Config":
        """Return a config that uses all visible local GPUs for full model paths."""
        from dataclasses import replace

        cfg = replace(
            self,
            require_gpu_for_full_run=require_gpu,
            lightning_accelerator="gpu" if require_gpu else "auto",
            lightning_devices=lightning_devices,
            lightning_strategy=lightning_strategy,
            chronos_device_map=chronos_device_map,
            chronos_parallel_devices=lightning_devices,
        )
        cfg.validate()
        return cfg

    def neuralforecast_trainer_kwargs(self) -> dict[str, Any]:
        kwargs: dict[str, Any] = {
            "accelerator": self.lightning_accelerator,
            "devices": self.lightning_devices,
        }
        if self.lightning_strategy is not None:
            kwargs["strategy"] = self.lightning_strategy
        return kwargs

    def targets_to_run(self, targets: Sequence[str] | None = None) -> tuple[str, ...]:
        selected = tuple(targets) if targets is not None else self.targets
        unknown = [t for t in selected if t not in self.targets]
        if unknown:
            raise ValueError(f"Unknown target(s) not in v210 target set: {unknown}")
        if not selected:
            raise ValueError("At least one target is required.")
        return selected

    def validate(self) -> None:
        if self.horizon < 1:
            raise ValueError("horizon must be positive.")
        if self.test_size < 1:
            raise ValueError("test_size must be positive.")
        if self.step_size < 1:
            raise ValueError("step_size must be positive.")
        if self.input_size < 1:
            raise ValueError("input_size must be positive.")
        if self.val_check_steps < 1:
            raise ValueError("val_check_steps must be positive.")
        if not self.proposed_k_grid or any(k < 1 for k in self.proposed_k_grid):
            raise ValueError("proposed_k_grid must contain positive integers.")
        if not 0.0 <= self.proposed_motif_weight <= 1.0:
            raise ValueError("proposed_motif_weight must be in [0, 1].")
        if self.proposed_temperature <= 0:
            raise ValueError("proposed_temperature must be positive.")
        if self.lightning_accelerator not in {"auto", "cpu", "gpu", "cuda", "mps", "tpu"}:
            raise ValueError(f"Unsupported lightning_accelerator: {self.lightning_accelerator}")
        if isinstance(self.lightning_devices, int) and self.lightning_devices == 0:
            raise ValueError("lightning_devices must not be 0; use -1 for all GPUs or a positive count.")
        if not self.chronos_device_map:
            raise ValueError("chronos_device_map must be non-empty.")


def default_config() -> V210Config:
    cfg = V210Config()
    cfg.validate()
    return cfg


# ===== Embedded RAG4CTS/data.py =====
from pathlib import Path

import numpy as np
import pandas as pd



def normalize_dt(s: pd.Series) -> pd.Series:
    """Match notebook date normalization: replace dots with dashes and coerce."""
    return pd.to_datetime(s.astype(str).str.replace(".", "-", regex=False), errors="coerce")


def load_weekly_csv(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"CSV not found: {path}")
    df = pd.read_csv(path)
    if "dt" not in df.columns:
        raise KeyError("CSV must contain 'dt'.")
    df["dt"] = normalize_dt(df["dt"])
    bad_dates = int(df["dt"].isna().sum())
    if bad_dates == len(df):
        raise ValueError("All dt values failed to parse.")
    df = df.dropna(subset=["dt"]).sort_values("dt").drop_duplicates("dt").reset_index(drop=True)
    for c in df.columns:
        if c != "dt":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def enforce_2011_variable_policy(df_raw: pd.DataFrame, cfg: V210Config) -> tuple[pd.DataFrame, pd.DataFrame]:
    cfg.validate()
    if "dt" not in df_raw.columns:
        raise KeyError("DataFrame must contain 'dt'.")
    missing_targets = [t for t in cfg.targets if t not in df_raw.columns]
    if missing_targets:
        raise KeyError(f"Missing target columns: {missing_targets}")

    start_rows = df_raw.index[df_raw["dt"].eq(cfg.fixed_start)].tolist()
    if not start_rows:
        raise ValueError(f"{cfg.fixed_start.date()} not found in dt column.")
    start_idx = start_rows[0]

    audit_rows: list[dict[str, object]] = []
    keep_cols = ["dt"]
    numeric_cols = [c for c in df_raw.columns if c != "dt" and pd.api.types.is_numeric_dtype(df_raw[c])]

    for col in numeric_cols:
        s = df_raw[col]
        first_valid_idx = s.first_valid_index()
        first_valid_dt = pd.Timestamp(df_raw.loc[first_valid_idx, "dt"]) if first_valid_idx is not None else pd.NaT
        v0 = s.iloc[start_idx]
        finite_at_start = bool(pd.notna(v0) and np.isfinite(float(v0)))

        if col in cfg.targets:
            keep = finite_at_start
            reason = "target_finite_at_start" if keep else "target_missing_at_start"
        else:
            keep = bool(finite_at_start and pd.notna(first_valid_dt) and first_valid_dt <= cfg.fixed_start)
            reason = "kept_finite_at_2011_01_03" if keep else "dropped_missing_at_2011_01_03"

        audit_rows.append({
            "column": col,
            "is_target": col in cfg.targets,
            "first_valid_dt": None if pd.isna(first_valid_dt) else first_valid_dt.strftime("%Y-%m-%d"),
            "finite_at_fixed_start": finite_at_start,
            "value_at_fixed_start": float(v0) if finite_at_start else np.nan,
            "kept": keep,
            "reason": reason,
        })
        if keep:
            keep_cols.append(col)

    audit = pd.DataFrame(audit_rows)
    bad_targets = audit[(audit.is_target) & (~audit.kept)]["column"].tolist()
    if bad_targets:
        raise ValueError(f"Target missing at fixed start: {bad_targets}")

    df = df_raw.loc[df_raw["dt"].ge(cfg.fixed_start), keep_cols].copy().reset_index(drop=True)
    value_cols = [c for c in df.columns if c != "dt"]
    if not value_cols:
        raise ValueError("No value columns retained after fixed-start policy.")

    missing_first = df.loc[0, value_cols].isna()
    if missing_first.any():
        raise ValueError(f"Kept columns missing at fixed start: {missing_first[missing_first].index.tolist()}")

    df[value_cols] = df[value_cols].ffill()
    remaining = df[value_cols].isna().sum()
    if (remaining > 0).any():
        raise ValueError(f"Missing remains after causal ffill: {remaining[remaining > 0].to_dict()}")

    return df, audit.sort_values(["is_target", "kept", "column"], ascending=[False, False, True]).reset_index(drop=True)


def load_selected_weekly_data(path: str | Path, cfg: V210Config) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    raw = load_weekly_csv(path)
    selected, audit = enforce_2011_variable_policy(raw, cfg)
    return raw, selected, audit


# ===== Embedded RAG4CTS/transforms.py =====
import math

import numpy as np
import pandas as pd



def validate_positive_targets(df: pd.DataFrame, cfg: V210Config, targets: tuple[str, ...] | None = None) -> None:
    for t in cfg.targets_to_run(targets):
        if t not in df.columns:
            raise KeyError(f"Missing target column: {t}")
        bad_mask = df[t].astype(float) <= 0
        if bad_mask.any():
            bad = df.loc[bad_mask, ["dt", t]].head()
            raise ValueError(f"{t} has non-positive weekly price; log-return transform invalid. First bad:\n{bad}")

def weekly_log_return(df: pd.DataFrame, target: str) -> np.ndarray:
    if target not in df.columns:
        raise KeyError(f"Missing target column: {target}")
    p = df[target].astype(float).to_numpy()
    if np.any(p <= 0):
        raise ValueError(f"{target} contains non-positive values; cannot compute log returns.")
    logp = np.log(p)
    r = np.zeros_like(logp)
    r[1:] = np.diff(logp)
    return r


def target_returns(df: pd.DataFrame, cfg: V210Config, targets: tuple[str, ...] | None = None) -> dict[str, np.ndarray]:
    selected = cfg.targets_to_run(targets)
    validate_positive_targets(df, cfg, selected)
    return {t: weekly_log_return(df, t) for t in selected}


def prices_from_returns(origin_price: float, future_returns: list[float] | tuple[float, ...] | np.ndarray) -> list[float]:
    if origin_price <= 0:
        raise ValueError("origin_price must be positive.")
    out: list[float] = []
    cumulative = 0.0
    for ret in future_returns:
        cumulative += float(ret)
        out.append(float(origin_price * math.exp(cumulative)))
    return out


def price_from_returns(origin_price: float, r1: float, r2: float) -> tuple[float, float]:
    """Backward-compatible two-step price conversion."""
    h1, h2 = prices_from_returns(origin_price, [r1, r2])
    return h1, h2


def make_neuralforecast_df(df: pd.DataFrame, returns: dict[str, np.ndarray], target: str) -> pd.DataFrame:
    if target not in returns:
        raise KeyError(f"Missing returns for target: {target}")
    if len(returns[target]) != len(df):
        raise ValueError("Return vector length must match df length.")
    return pd.DataFrame({"unique_id": target, "ds": df["dt"], "y": returns[target]})


# ===== Embedded RAG4CTS/panel.py =====
import pandas as pd



def date_to_idx(df: pd.DataFrame) -> dict[str, int]:
    if "dt" not in df.columns:
        raise KeyError("DataFrame must contain 'dt'.")
    return {pd.Timestamp(d).strftime("%Y-%m-%d"): int(i) for i, d in enumerate(df["dt"])}


def evaluation_origin_indices(df: pd.DataFrame, cfg: V210Config, max_origins: int | None = None) -> list[int]:
    cfg.validate()
    if "dt" not in df.columns:
        raise KeyError("DataFrame must contain 'dt'.")
    out: list[int] = []
    for i, d in enumerate(df["dt"]):
        d = pd.Timestamp(d)
        if d >= cfg.origin_start and i + cfg.horizon < len(df) and i - cfg.input_size + 1 >= 0:
            out.append(int(i))
    if max_origins is not None:
        if max_origins < 1:
            raise ValueError("max_origins must be positive when provided.")
        out = out[: int(max_origins)]
    if not out:
        raise ValueError("No evaluation origins available for the configured origin_start/input_size/horizon.")
    return out


def origin_dates(df: pd.DataFrame, origin_indices: list[int]) -> list[str]:
    return [pd.Timestamp(df.loc[i, "dt"]).strftime("%Y-%m-%d") for i in origin_indices]


def build_origin_panel(
    df: pd.DataFrame,
    cfg: V210Config,
    targets_to_run: tuple[str, ...] | None = None,
    max_origins: int | None = None,
) -> tuple[pd.DataFrame, list[int], list[str]]:
    targets = cfg.targets_to_run(targets_to_run)
    missing = [t for t in targets if t not in df.columns]
    if missing:
        raise KeyError(f"Missing target columns: {missing}")
    indices = evaluation_origin_indices(df, cfg, max_origins=max_origins)
    rows: list[dict[str, object]] = []
    for oi in indices:
        for target in targets:
            row: dict[str, object] = {
                "panel": cfg.panel_name,
                "origin_date": pd.Timestamp(df.loc[oi, "dt"]).strftime("%Y-%m-%d"),
                "origin_idx": int(oi),
                "target": target,
                "origin_price": float(df.loc[oi, target]),
            }
            for h in range(1, cfg.horizon + 1):
                row[f"h{h}_date"] = pd.Timestamp(df.loc[oi + h, "dt"]).strftime("%Y-%m-%d")
                row[f"actual_h{h}_price"] = float(df.loc[oi + h, target])
            rows.append(row)
    panel = pd.DataFrame(rows)
    if panel.empty:
        raise ValueError("Origin panel is empty.")
    return panel, indices, origin_dates(df, indices)


# ===== Embedded RAG4CTS/gpu.py =====
from typing import Any



def torch_gpu_summary() -> dict[str, Any]:
    """Return local CUDA/GPU details without making torch a package import dependency."""
    try:
        import torch
    except ImportError:
        return {"torch_available": False, "cuda_available": False, "gpu_count": 0, "gpus": []}

    cuda_available = bool(torch.cuda.is_available())
    count = int(torch.cuda.device_count()) if cuda_available else 0
    gpus = []
    for idx in range(count):
        props = torch.cuda.get_device_properties(idx)
        gpus.append({
            "index": idx,
            "name": props.name,
            "total_memory_gb": round(float(props.total_memory) / 1024**3, 2),
        })
    return {
        "torch_available": True,
        "torch_version": getattr(torch, "__version__", None),
        "cuda_available": cuda_available,
        "cuda_version": getattr(torch.version, "cuda", None),
        "gpu_count": count,
        "gpus": gpus,
    }


def validate_torch_gpu_runtime(torch_module: Any, cfg: V210Config, *, caller: str) -> None:
    """Fast-fail when a full model path is configured to require local CUDA."""
    if not cfg.require_gpu_for_full_run:
        return
    if not bool(torch_module.cuda.is_available()):
        raise RuntimeError(
            f"{caller} requires CUDA because require_gpu_for_full_run=True. "
            "Run on a GPU machine or set require_gpu_for_full_run=False explicitly."
        )
    if int(torch_module.cuda.device_count()) < 1:
        raise RuntimeError(f"{caller} requires at least one CUDA device.")


# ===== Embedded RAG4CTS/chronos_runner.py =====
from typing import Any

import pandas as pd


BASE_TIME = pd.Timestamp("2000-01-03")


def load_chronos_pipeline(cfg: V210Config):
    try:
        import torch
        from chronos import Chronos2Pipeline
    except ImportError as exc:
        raise ImportError("Chronos execution requires torch and chronos-forecasting>=2.0.") from exc
    validate_torch_gpu_runtime(torch, cfg, caller="Chronos full run")
    device_map = cfg.chronos_device_map if torch.cuda.is_available() else "cpu"
    kwargs: dict[str, Any] = {"device_map": device_map}
    if cfg.chronos_dtype is not None:
        kwargs["dtype"] = cfg.chronos_dtype
    pipe = Chronos2Pipeline.from_pretrained(cfg.chronos_model_id, **kwargs)
    return pipe


def _median_output(pred_df: pd.DataFrame, item_id: str = "query") -> tuple[pd.Series, str]:
    q = pred_df[pred_df["item_id"].astype(str).eq(item_id)].sort_values("timestamp")
    if q.empty:
        raise RuntimeError(f"Chronos returned no rows for item_id={item_id!r}.")
    for col in ["predictions", "0.5", "q0.5", "median", "mean", "target"]:
        if col in q.columns:
            return q[col].astype(float), col
    numeric_cols = [c for c in q.columns if c not in ["item_id", "timestamp", "target_name"] and pd.api.types.is_numeric_dtype(q[c])]
    if not numeric_cols:
        raise RuntimeError(f"Cannot identify Chronos output column. Columns={list(q.columns)}")
    return q[numeric_cols[0]].astype(float), numeric_cols[0]


def chronos_predict_return_path_with_raw(pipe, history_returns: list[float], cfg: V210Config) -> tuple[list[float], dict[str, Any], pd.DataFrame]:
    if len(history_returns) < 1:
        raise ValueError("history_returns must not be empty.")
    context_df = pd.DataFrame({
        "item_id": "query",
        "timestamp": [BASE_TIME + pd.Timedelta(weeks=i) for i in range(len(history_returns))],
        "target": [float(x) for x in history_returns],
    })
    try:
        pred_df = pipe.predict_df(context_df, prediction_length=cfg.horizon, quantile_levels=list(cfg.quantile_levels), id_column="item_id", timestamp_column="timestamp", target="target", validate_inputs=False, freq="W-MON")
    except TypeError:
        pred_df = pipe.predict_df(context_df, prediction_length=cfg.horizon, quantile_levels=list(cfg.quantile_levels), id_column="item_id", timestamp_column="timestamp", target="target")
    vals, value_col = _median_output(pred_df)
    arr = vals.to_numpy()
    if len(arr) < cfg.horizon:
        raise RuntimeError(f"Chronos returned fewer than {cfg.horizon} steps.")
    return [float(x) for x in arr[: cfg.horizon]], {"chronos_value_col": value_col, "chronos_rows": int(len(vals))}, pred_df


def chronos_predict_returns_with_raw(pipe, history_returns: list[float], cfg: V210Config) -> tuple[float, float, dict[str, Any], pd.DataFrame]:
    """Backward-compatible two-step Chronos helper."""
    path, meta, pred_df = chronos_predict_return_path_with_raw(pipe, history_returns, cfg)
    if len(path) < 2:
        raise RuntimeError("Chronos returned fewer than 2 steps.")
    return path[0], path[1], meta, pred_df


def chronos_predict_return_path(pipe, history_returns: list[float], cfg: V210Config) -> tuple[list[float], dict[str, Any]]:
    path, meta, _pred_df = chronos_predict_return_path_with_raw(pipe, history_returns, cfg)
    return path, meta


def chronos_predict_returns(pipe, history_returns: list[float], cfg: V210Config) -> tuple[float, float, dict[str, Any]]:
    r1, r2, meta, _pred_df = chronos_predict_returns_with_raw(pipe, history_returns, cfg)
    return r1, r2, meta


def chronos_predict_future_path_with_raw(pipe, context_df: pd.DataFrame, future_df: pd.DataFrame, cfg: V210Config) -> tuple[list[float], dict[str, Any], pd.DataFrame]:
    try:
        pred_df = pipe.predict_df(context_df, future_df=future_df, prediction_length=cfg.horizon, quantile_levels=list(cfg.quantile_levels), id_column="item_id", timestamp_column="timestamp", target="target", validate_inputs=False, freq="W-MON")
    except TypeError:
        pred_df = pipe.predict_df(context_df, future_df=future_df, prediction_length=cfg.horizon, quantile_levels=list(cfg.quantile_levels), id_column="item_id", timestamp_column="timestamp", target="target")
    vals, value_col = _median_output(pred_df)
    arr = vals.to_numpy()
    if len(arr) < cfg.horizon:
        raise RuntimeError(f"Chronos returned fewer than {cfg.horizon} steps.")
    return [float(x) for x in arr[: cfg.horizon]], {"chronos_value_col": value_col, "chronos_rows": int(len(vals))}, pred_df


def chronos_predict_with_future_df_with_raw(pipe, context_df: pd.DataFrame, future_df: pd.DataFrame, cfg: V210Config) -> tuple[float, float, dict[str, Any], pd.DataFrame]:
    """Backward-compatible two-step Chronos helper with future_df."""
    path, meta, pred_df = chronos_predict_future_path_with_raw(pipe, context_df, future_df, cfg)
    if len(path) < 2:
        raise RuntimeError("Chronos returned fewer than 2 steps.")
    return path[0], path[1], meta, pred_df


def chronos_predict_future_path(pipe, context_df: pd.DataFrame, future_df: pd.DataFrame, cfg: V210Config) -> tuple[list[float], dict[str, Any]]:
    path, meta, _pred_df = chronos_predict_future_path_with_raw(pipe, context_df, future_df, cfg)
    return path, meta


def chronos_predict_with_future_df(pipe, context_df: pd.DataFrame, future_df: pd.DataFrame, cfg: V210Config) -> tuple[float, float, dict[str, Any]]:
    r1, r2, meta, _pred_df = chronos_predict_with_future_df_with_raw(pipe, context_df, future_df, cfg)
    return r1, r2, meta


# ===== Embedded RAG4CTS/neuralforecast_runner.py =====
import gc
from typing import Any

import pandas as pd



def _require_neuralforecast():
    try:
        import torch
        from torch.optim import AdamW
        from torch.optim.lr_scheduler import ReduceLROnPlateau
        from neuralforecast import NeuralForecast
        from neuralforecast.models import GRU, TimeXer, iTransformer
        from neuralforecast.losses.pytorch import MSE
    except ImportError as exc:
        raise ImportError(
            "NeuralForecast execution requires torch, neuralforecast, and pytorch-lightning. "
            "Install notebook dependencies before running the full model path."
        ) from exc
    return torch, AdamW, ReduceLROnPlateau, NeuralForecast, GRU, TimeXer, iTransformer, MSE


def build_nf_models(cfg: V210Config) -> list[Any]:
    torch, AdamW, ReduceLROnPlateau, _NeuralForecast, GRU, TimeXer, iTransformer, MSE = _require_neuralforecast()
    validate_torch_gpu_runtime(torch, cfg, caller="NeuralForecast full run")

    def common_nf_kwargs(alias: str) -> dict[str, Any]:
        return dict(
            h=cfg.horizon,
            input_size=cfg.input_size,
            loss=MSE(),
            valid_loss=MSE(),
            max_steps=cfg.max_steps,
            learning_rate=cfg.learning_rate,
            early_stop_patience_steps=cfg.early_stop_patience_steps,
            batch_size=cfg.batch_size,
            valid_batch_size=cfg.valid_batch_size,
            windows_batch_size=cfg.windows_batch_size,
            inference_windows_batch_size=cfg.inference_windows_batch_size,
            step_size=cfg.step_size,
            val_check_steps=cfg.val_check_steps,
            scaler_type=cfg.scaler_type,
            random_seed=cfg.seed,
            alias=alias,
            enable_progress_bar=True,
            logger=False,
            **cfg.neuralforecast_trainer_kwargs(),
        )

    class TimeXerAdamWPlateau(TimeXer):
        def configure_optimizers(self):
            optimizer = AdamW(self.parameters(), lr=cfg.learning_rate)
            scheduler = ReduceLROnPlateau(optimizer, **cfg.scheduler_kwargs)
            return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "monitor": "valid_loss", "interval": "epoch", "frequency": 1, "strict": False}}

    class ITransformerAdamWPlateau(iTransformer):
        def configure_optimizers(self):
            optimizer = AdamW(self.parameters(), lr=cfg.learning_rate)
            scheduler = ReduceLROnPlateau(optimizer, **cfg.scheduler_kwargs)
            return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "monitor": "valid_loss", "interval": "epoch", "frequency": 1, "strict": False}}

    class GRUAdamWPlateau(GRU):
        def configure_optimizers(self):
            optimizer = AdamW(self.parameters(), lr=cfg.learning_rate)
            scheduler = ReduceLROnPlateau(optimizer, **cfg.scheduler_kwargs)
            return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "monitor": "valid_loss", "interval": "epoch", "frequency": 1, "strict": False}}

    return [
        TimeXerAdamWPlateau(
            **common_nf_kwargs("TimeXer"), n_series=1, patch_len=16, hidden_size=256, n_heads=8,
            e_layers=2, d_ff=1024, factor=1, dropout=0.1, use_norm=True, exclude_insample_y=False,
        ),
        ITransformerAdamWPlateau(
            **common_nf_kwargs("iTransformer"), n_series=1, hidden_size=256, n_heads=8,
            e_layers=2, d_layers=1, d_ff=1024, factor=1, dropout=0.1, use_norm=True, exclude_insample_y=False,
        ),
        GRUAdamWPlateau(
            **common_nf_kwargs("GRU"), encoder_n_layers=2, encoder_hidden_size=400, encoder_bias=True,
            encoder_dropout=0.0, decoder_hidden_size=256, decoder_layers=2, h_train=1,
            recurrent=False, exclude_insample_y=False,
        ),
    ]


def convert_nf_cv_to_price_predictions(
    cv_df: pd.DataFrame,
    target: str,
    model_cols: list[str],
    df: pd.DataFrame,
    cfg: V210Config,
    eval_origin_dates: list[str],
) -> pd.DataFrame:
    out_rows: list[dict[str, object]] = []
    tmp = cv_df.copy()
    required = {"cutoff", "ds", *model_cols}
    missing = [c for c in required if c not in tmp.columns]
    if missing:
        raise KeyError(f"NeuralForecast CV output missing columns: {missing}")
    index_by_date = date_to_idx(df)
    tmp["cutoff"] = pd.to_datetime(tmp["cutoff"]).dt.strftime("%Y-%m-%d")
    tmp["ds"] = pd.to_datetime(tmp["ds"]).dt.strftime("%Y-%m-%d")
    tmp = tmp[tmp["cutoff"].isin(eval_origin_dates)].copy()

    for cutoff, g in tmp.groupby("cutoff"):
        if cutoff not in index_by_date:
            continue
        oi = index_by_date[cutoff]
        origin_price = float(df.loc[oi, target])
        g = g.sort_values("ds")
        if len(g) < cfg.horizon:
            continue
        for model in model_cols:
            future_returns = [float(g.iloc[h - 1][model]) for h in range(1, cfg.horizon + 1)]
            pred_prices = prices_from_returns(origin_price, future_returns)
            cumulative_return = 0.0
            for horizon, (pred_price, step_return) in enumerate(zip(pred_prices, future_returns), start=1):
                cumulative_return += float(step_return)
                actual_price = float(df.loc[oi + horizon, target])
                out_rows.append({
                    "panel": cfg.panel_name, "model": model, "origin_date": cutoff, "target": target,
                    "horizon": horizon, "origin_price": origin_price, "pred_price": float(pred_price),
                    "actual_price": float(actual_price), "abs_error": abs(float(pred_price) - float(actual_price)),
                    "pred_return_step": float(step_return),
                    "pred_return_from_origin": float(cumulative_return),
                    "actual_delta_from_origin": float(actual_price) - origin_price,
                    "pred_delta_from_origin": float(pred_price) - origin_price,
                })
    return pd.DataFrame(out_rows)


def run_neuralforecast_tscv(df: pd.DataFrame, returns: dict[str, object], cfg: V210Config, eval_origin_dates: list[str], targets_to_run: tuple[str, ...]) -> tuple[pd.DataFrame, pd.DataFrame]:
    torch, _AdamW, _ReduceLROnPlateau, NeuralForecast, *_rest = _require_neuralforecast()
    validate_torch_gpu_runtime(torch, cfg, caller="NeuralForecast full run")
    pred_parts: list[pd.DataFrame] = []
    raw_parts: list[pd.DataFrame] = []
    for target in targets_to_run:
        y_df = make_neuralforecast_df(df, returns, target)
        nf = NeuralForecast(models=build_nf_models(cfg), freq="W-MON")
        cv_df = nf.cross_validation(df=y_df, n_windows=len(eval_origin_dates), step_size=cfg.step_size, val_size=cfg.val_size, refit=1, verbose=1)
        cv_df["target_name"] = target
        raw_parts.append(cv_df)
        model_cols = [c for c in ["TimeXer", "iTransformer", "GRU"] if c in cv_df.columns]
        pred_parts.append(convert_nf_cv_to_price_predictions(cv_df, target, model_cols, df, cfg, eval_origin_dates))
        del nf
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return (
        pd.concat(pred_parts, ignore_index=True) if pred_parts else pd.DataFrame(),
        pd.concat(raw_parts, ignore_index=True) if raw_parts else pd.DataFrame(),
    )


# ===== Embedded RAG4CTS/retrieval.py =====
from typing import Any, Sequence

import numpy as np
import pandas as pd



def robust_zscore(arr: np.ndarray) -> np.ndarray:
    med = np.nanmedian(arr, axis=0)
    q25 = np.nanpercentile(arr, 25, axis=0)
    q75 = np.nanpercentile(arr, 75, axis=0)
    scale = (q75 - q25) / 1.349
    std = np.nanstd(arr, axis=0)
    scale = np.where(np.isfinite(scale) & (np.abs(scale) > 1e-12), scale, std)
    scale = np.where(np.isfinite(scale) & (np.abs(scale) > 1e-12), scale, 1.0)
    return np.nan_to_num(np.clip((arr - med) / scale, -10, 10), nan=0.0, posinf=0.0, neginf=0.0)


def covariate_feature_matrix(df: pd.DataFrame, cfg: V210Config) -> np.ndarray:
    cov_cols = [c for c in df.columns if c not in ["dt", *cfg.targets]]
    if cov_cols:
        cov_level = df[cov_cols].astype(float).to_numpy()
        cov_diff = np.vstack([np.zeros((1, cov_level.shape[1])), np.diff(cov_level, axis=0)])
        return np.concatenate([robust_zscore(cov_level), robust_zscore(cov_diff)], axis=1)
    return np.zeros((len(df), 1), dtype=float)


class ProposedRetriever:
    """Strict-parity retrieval helpers for the proposed dynamic-K Chronos path."""

    def __init__(self, df: pd.DataFrame, target_returns: dict[str, np.ndarray], cfg: V210Config):
        self.df = df
        self.target_returns = target_returns
        self.cfg = cfg
        self.cov_feature = covariate_feature_matrix(df, cfg)

    def _require_target(self, target: str) -> np.ndarray:
        if target not in self.target_returns:
            raise KeyError(f"Missing target returns for {target}")
        return self.target_returns[target]

    def target_motif(self, origin_idx: int, target: str, input_weeks: int | None = None) -> np.ndarray:
        weeks = int(input_weeks or self.cfg.proposed_input_weeks)
        start = origin_idx - weeks + 1
        if start < 1:
            raise ValueError("insufficient target motif history")
        r = self._require_target(target)[start : origin_idx + 1].copy()
        mu, sd = float(np.mean(r)), float(np.std(r))
        if not np.isfinite(sd) or sd < 1e-12:
            sd = 1.0
        return ((r - mu) / sd).astype(float)

    def covariate_motif(self, origin_idx: int, input_weeks: int | None = None) -> np.ndarray:
        weeks = int(input_weeks or self.cfg.proposed_input_weeks)
        start = origin_idx - weeks + 1
        if start < 0:
            raise ValueError("insufficient covariate motif history")
        return self.cov_feature[start : origin_idx + 1].reshape(-1).astype(float)

    @staticmethod
    def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
        denom = max(float(np.linalg.norm(a) * np.linalg.norm(b)), 1e-12)
        return float(np.dot(a, b) / denom)

    def candidate_indices_for_query(self, query_origin_idx: int, input_weeks: int | None = None) -> list[int]:
        weeks = int(input_weeks or self.cfg.proposed_input_weeks)
        query_input_start = query_origin_idx - weeks + 1
        out: list[int] = []
        for ci in range(weeks, len(self.df) - self.cfg.horizon):
            if ci + self.cfg.horizon >= query_input_start:
                continue
            out.append(int(ci))
        return out

    def score_candidates(self, origin_idx: int, target: str, motif_weight: float | None = None) -> list[dict[str, Any]]:
        weight = self.cfg.proposed_motif_weight if motif_weight is None else float(motif_weight)
        q_target = self.target_motif(origin_idx, target)
        q_cov = self.covariate_motif(origin_idx)
        returns = self._require_target(target)
        out: list[dict[str, Any]] = []
        for ci in self.candidate_indices_for_query(origin_idx):
            target_sim = self.cosine_similarity(q_target, self.target_motif(ci, target))
            cov_sim = self.cosine_similarity(q_cov, self.covariate_motif(ci))
            score = float(weight * target_sim + (1.0 - weight) * cov_sim)
            out.append({
                "candidate_origin_idx": int(ci),
                "candidate_origin_date": pd.Timestamp(self.df.loc[ci, "dt"]).strftime("%Y-%m-%d"),
                "target_similarity": float(target_sim),
                "covariate_similarity": float(cov_sim),
                "blend_score": score,
            })
            for h in range(1, self.cfg.horizon + 1):
                out[-1][f"candidate_h{h}_date"] = pd.Timestamp(self.df.loc[ci + h, "dt"]).strftime("%Y-%m-%d")
                out[-1][f"future_r{h}"] = float(returns[ci + h])
        return sorted(out, key=lambda r: r["blend_score"], reverse=True)

    def softmax_scores(self, scores: np.ndarray, temp: float | None = None) -> np.ndarray:
        temperature = self.cfg.proposed_temperature if temp is None else float(temp)
        if temperature <= 0:
            raise ValueError("temperature must be positive")
        s = np.asarray(scores, dtype=float) / max(temperature, 1e-12)
        s -= np.max(s)
        w = np.exp(s)
        return w / max(float(w.sum()), 1e-12)

    def aggregate_future_path(self, top: list[dict[str, Any]]) -> tuple[list[float], list[dict[str, Any]]]:
        if not top:
            return [float("nan")] * self.cfg.horizon, []
        w = self.softmax_scores(np.array([r["blend_score"] for r in top], dtype=float))
        weighted: list[dict[str, Any]] = []
        for wi, row in zip(w, top):
            rr = dict(row)
            rr["retrieval_weight"] = float(wi)
            weighted.append(rr)
        path = [
            float(np.dot(w, np.array([r[f"future_r{h}"] for r in top], dtype=float)))
            for h in range(1, self.cfg.horizon + 1)
        ]
        return path, weighted

    def aggregate_future(self, top: list[dict[str, Any]]) -> tuple[float, float, list[dict[str, Any]]]:
        """Backward-compatible two-step aggregate helper."""
        path, weighted = self.aggregate_future_path(top)
        if len(path) < 2:
            raise RuntimeError("aggregate_future requires horizon >= 2.")
        return path[0], path[1], weighted

    def choose_dynamic_k(self, origin_idx: int, target: str) -> tuple[int, pd.DataFrame, dict[str, Any]]:
        candidates = self.score_candidates(origin_idx, target)
        if not candidates:
            return self.cfg.proposed_k_grid[0], pd.DataFrame(), {"pseudo_available": False, "reason": "no_candidates"}
        pseudo_idx = int(candidates[0]["candidate_origin_idx"])
        pseudo_candidates = self.score_candidates(pseudo_idx, target)
        returns = self._require_target(target)
        rows: list[dict[str, Any]] = []
        for k in self.cfg.proposed_k_grid:
            top = pseudo_candidates[: min(int(k), len(pseudo_candidates))]
            pred_path, _ = self.aggregate_future_path(top)
            row: dict[str, Any] = {
                "origin_idx": int(origin_idx),
                "origin_date": pd.Timestamp(self.df.loc[origin_idx, "dt"]).strftime("%Y-%m-%d"),
                "target": target,
                "pseudo_origin_idx": int(pseudo_idx),
                "pseudo_origin_date": pd.Timestamp(self.df.loc[pseudo_idx, "dt"]).strftime("%Y-%m-%d"),
                "k": int(k),
                "pseudo_candidate_count": int(len(pseudo_candidates)),
            }
            losses: list[float] = []
            pred_cum = 0.0
            true_cum = 0.0
            for h, pred_r in enumerate(pred_path, start=1):
                pred_cum += float(pred_r)
                true_cum += float(returns[pseudo_idx + h])
                h_loss = abs(pred_cum - true_cum)
                row[f"pseudo_h{h}_loss"] = float(h_loss)
                losses.append(float(h_loss))
            row["pseudo_loss_return_mae"] = float(np.mean(losses)) if losses else float("nan")
            rows.append(row)
        audit = pd.DataFrame(rows).sort_values(["pseudo_loss_return_mae", "k"]).reset_index(drop=True)
        selected_k = int(audit.iloc[0]["k"])
        audit["selected"] = audit["k"].eq(selected_k)
        return selected_k, audit, {
            "pseudo_available": True,
            "pseudo_origin_idx": int(pseudo_idx),
            "pseudo_origin_date": pd.Timestamp(self.df.loc[pseudo_idx, "dt"]).strftime("%Y-%m-%d"),
            "selected_k": selected_k,
            "selected_pseudo_loss_return_mae": float(audit.iloc[0]["pseudo_loss_return_mae"]),
        }

    def build_proposed_chronos_context(
        self,
        origin_idx: int,
        target: str,
        top_contexts: list[dict[str, Any]],
        proxy_returns: Sequence[float],
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
        returns = self._require_target(target)
        parts: list[pd.DataFrame] = []
        q_start = origin_idx - self.cfg.proposed_input_weeks + 1
        if q_start < 0:
            raise ValueError("insufficient query history")
        q_values = returns[q_start : origin_idx + 1].tolist()
        parts.append(pd.DataFrame({
            "item_id": "query",
            "timestamp": [BASE_TIME + pd.Timedelta(weeks=i) for i in range(len(q_values))],
            "target": [float(x) for x in q_values],
            "raft_future_return": 0.0,
            "is_query": 1,
        }))
        for j, ctx in enumerate(top_contexts):
            ci = int(ctx["candidate_origin_idx"])
            c_start = ci - self.cfg.proposed_input_weeks + 1
            vals = returns[c_start : ci + self.cfg.horizon + 1].tolist()
            parts.append(pd.DataFrame({
                "item_id": f"ctx_{j:02d}",
                "timestamp": [BASE_TIME + pd.Timedelta(weeks=i) for i in range(len(vals))],
                "target": [float(x) for x in vals],
                "raft_future_return": 0.0,
                "is_query": 0,
            }))
        context_df = pd.concat(parts, ignore_index=True)
        future_parts: list[pd.DataFrame] = []
        for item_id, g in context_df.groupby("item_id"):
            last_ts = g["timestamp"].max()
            if item_id == "query" and self.cfg.proposed_use_future_covariate:
                vals = [float(x) for x in proxy_returns[: self.cfg.horizon]]
                if len(vals) < self.cfg.horizon:
                    vals.extend([0.0] * (self.cfg.horizon - len(vals)))
                is_query = 1
            else:
                vals = [0.0] * self.cfg.horizon
                is_query = 0
            future_parts.append(pd.DataFrame({
                "item_id": item_id,
                "timestamp": [last_ts + pd.Timedelta(weeks=i) for i in range(1, self.cfg.horizon + 1)],
                "raft_future_return": vals,
                "is_query": is_query,
            }))
        return (
            context_df.sort_values(["item_id", "timestamp"]).reset_index(drop=True),
            pd.concat(future_parts, ignore_index=True).sort_values(["item_id", "timestamp"]).reset_index(drop=True),
        )


# ===== Embedded RAG4CTS/full_runner.py =====
from typing import Any, Sequence

import pandas as pd



CHRONOS_QUERY_MODEL = "Chronos2_QueryOnly"
PROPOSED_MODEL = "Proposed_DynamicK_RetrievedContext_Chronos2"


def prediction_rows_from_return_path(
    *,
    cfg: V210Config,
    df: pd.DataFrame,
    target: str,
    origin_date: str,
    origin_idx: int,
    origin_price: float,
    model: str,
    future_returns: Sequence[float],
    extra: dict[str, Any] | None = None,
) -> list[dict[str, Any]]:
    pred_prices = prices_from_returns(origin_price, list(future_returns)[: cfg.horizon])
    rows: list[dict[str, Any]] = []
    cumulative_return = 0.0
    for h, (pred_price, step_return) in enumerate(zip(pred_prices, future_returns), start=1):
        if h > cfg.horizon:
            break
        cumulative_return += float(step_return)
        actual_price = float(df.loc[origin_idx + h, target])
        row: dict[str, Any] = {
            "panel": cfg.panel_name,
            "model": model,
            "origin_date": origin_date,
            "target": target,
            "horizon": h,
            "origin_price": float(origin_price),
            "pred_price": float(pred_price),
            "actual_price": actual_price,
            "abs_error": abs(float(pred_price) - actual_price),
            "pred_return_step": float(step_return),
            "pred_return_from_origin": float(cumulative_return),
            "actual_delta_from_origin": actual_price - float(origin_price),
            "pred_delta_from_origin": float(pred_price) - float(origin_price),
        }
        if extra:
            row.update(extra)
        rows.append(row)
    return rows


def run_chronos_query_tscv(
    df: pd.DataFrame,
    returns: dict[str, Any],
    cfg: V210Config,
    origin_panel: pd.DataFrame,
    targets_to_run: Sequence[str],
) -> pd.DataFrame:
    pipe = load_chronos_pipeline(cfg)
    rows: list[dict[str, Any]] = []
    for rec in origin_panel[origin_panel.target.isin(targets_to_run)].itertuples(index=False):
        origin_idx = int(rec.origin_idx)
        target = str(rec.target)
        history = returns[target][origin_idx - cfg.input_size + 1 : origin_idx + 1].tolist()
        path, meta = chronos_predict_return_path(pipe, history, cfg)
        rows.extend(
            prediction_rows_from_return_path(
                cfg=cfg,
                df=df,
                target=target,
                origin_date=str(rec.origin_date),
                origin_idx=origin_idx,
                origin_price=float(rec.origin_price),
                model=CHRONOS_QUERY_MODEL,
                future_returns=path,
                extra=meta,
            )
        )
    return pd.DataFrame(rows)


def run_proposed_tscv(
    df: pd.DataFrame,
    returns: dict[str, Any],
    cfg: V210Config,
    origin_panel: pd.DataFrame,
    targets_to_run: Sequence[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    pipe = load_chronos_pipeline(cfg)
    retriever = ProposedRetriever(df, returns, cfg)
    pred_rows: list[dict[str, Any]] = []
    audit_parts: list[pd.DataFrame] = []
    index_by_date = date_to_idx(df)

    for rec in origin_panel[origin_panel.target.isin(targets_to_run)].itertuples(index=False):
        origin_date = str(rec.origin_date)
        origin_idx = int(getattr(rec, "origin_idx", index_by_date[origin_date]))
        target = str(rec.target)
        selected_k, k_audit, k_meta = retriever.choose_dynamic_k(origin_idx, target)
        if not k_audit.empty:
            k_audit = k_audit.copy()
            k_audit.insert(0, "audit_type", "dynamic_k")
            audit_parts.append(k_audit)
        candidates = retriever.score_candidates(origin_idx, target)
        top = candidates[: min(selected_k, len(candidates))]
        proxy_path, weighted_contexts = retriever.aggregate_future_path(top)
        if weighted_contexts:
            donor_audit = pd.DataFrame(weighted_contexts)
            donor_audit.insert(0, "audit_type", "donor_context")
            donor_audit.insert(1, "origin_date", origin_date)
            donor_audit.insert(2, "origin_idx", origin_idx)
            donor_audit.insert(3, "target", target)
            donor_audit.insert(4, "selected_k", selected_k)
            audit_parts.append(donor_audit)
        context_df, future_df = retriever.build_proposed_chronos_context(
            origin_idx,
            target,
            weighted_contexts,
            proxy_returns=proxy_path,
        )
        path, meta = chronos_predict_future_path(pipe, context_df, future_df, cfg)
        pred_rows.extend(
            prediction_rows_from_return_path(
                cfg=cfg,
                df=df,
                target=target,
                origin_date=origin_date,
                origin_idx=origin_idx,
                origin_price=float(rec.origin_price),
                model=PROPOSED_MODEL,
                future_returns=path,
                extra={
                    **meta,
                    "selected_k": selected_k,
                    "candidate_count": len(candidates),
                    **{k: v for k, v in k_meta.items() if isinstance(v, (str, int, float, bool))},
                },
            )
        )

    audit = pd.concat(audit_parts, ignore_index=True) if audit_parts else pd.DataFrame()
    return pd.DataFrame(pred_rows), audit


# ===== Embedded RAG4CTS/metrics.py =====
import math

import numpy as np
import pandas as pd



def _metric_block(g: pd.DataFrame) -> dict[str, float]:
    actual = g["actual_price"].astype(float).to_numpy()
    pred = g["pred_price"].astype(float).to_numpy()
    err = pred - actual
    mape_denom = np.abs(actual)
    mape = np.where(mape_denom > 1e-12, 100.0 * np.abs(err) / mape_denom, np.nan)
    denom = np.abs(actual) + np.abs(pred)
    smape = np.where(denom > 1e-12, 200.0 * np.abs(err) / denom, 0.0)
    mse = float(np.mean(err ** 2)) if len(err) else np.nan
    return {
        "MSE": mse,
        "RMSE": float(math.sqrt(mse)) if np.isfinite(mse) else np.nan,
        "MAE": float(np.mean(np.abs(err))) if len(err) else np.nan,
        "MAPE": float(np.nanmean(mape)) if len(mape) else np.nan,
        "sMAPE": float(np.mean(smape)) if len(smape) else np.nan,
    }


def summarize_predictions(pred: pd.DataFrame, cfg: V210Config) -> pd.DataFrame:
    if pred.empty:
        return pd.DataFrame()
    required = {"model", "horizon", "origin_date", "target", "actual_price", "pred_price", "pred_delta_from_origin", "actual_delta_from_origin"}
    missing = [c for c in required if c not in pred.columns]
    if missing:
        raise KeyError(f"Prediction frame missing columns: {missing}")
    rows: list[dict[str, object]] = []
    for model, g in pred.groupby("model"):
        row: dict[str, object] = {
            "panel": cfg.panel_name,
            "model": model,
            "rows": int(len(g)),
            "origin_target_units": int(g[["origin_date", "target"]].drop_duplicates().shape[0]),
            "origins": int(g.origin_date.nunique()),
        }
        row.update({f"all_{k}": v for k, v in _metric_block(g).items()})
        row["all_sign_accuracy"] = float((np.sign(g.pred_delta_from_origin) == np.sign(g.actual_delta_from_origin)).mean())
        for h in range(1, cfg.horizon + 1):
            gh = g[g.horizon.eq(h)]
            for k, v in _metric_block(gh).items():
                row[f"h{h}_{k}"] = v
            row[f"h{h}_sign_accuracy"] = (
                float((np.sign(gh.pred_delta_from_origin) == np.sign(gh.actual_delta_from_origin)).mean())
                if len(gh)
                else np.nan
            )
        rows.append(row)
    return pd.DataFrame(rows).sort_values("all_MAPE").reset_index(drop=True)


def summarize_predictions_by_regime(pred: pd.DataFrame, cfg: V210Config) -> pd.DataFrame:
    if pred.empty or "regime" not in pred.columns:
        return pd.DataFrame()
    rows: list[dict[str, object]] = []
    group_cols = ["target", "regime", "model"]
    for (target, regime, model), g in pred.groupby(group_cols, dropna=False):
        row: dict[str, object] = {
            "panel": cfg.panel_name,
            "target": target,
            "regime": regime,
            "model": model,
            "rows": int(len(g)),
            "origins": int(g.origin_date.nunique()),
        }
        row.update(_metric_block(g))
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["target", "regime", "MAPE", "model"]).reset_index(drop=True)


def build_spike_labels(df: pd.DataFrame, origin_panel: pd.DataFrame, cfg: V210Config, rolling_weeks: int = 52, sigma: float = 3.0) -> pd.DataFrame:
    """Paper-aligned extreme movement labels.

    A forecast origin is labeled extreme when any realized weekly log return inside
    the forecast horizon exceeds the previous 52-week rolling mean by +/- 3 sigma.
    Labels are diagnostics only and are not used for model training, retrieval, or selection.
    """
    rows: list[dict[str, object]] = []
    for _, row in origin_panel.iterrows():
        target = str(row["target"])
        origin_idx = int(row["origin_idx"])
        returns = weekly_log_return(df, target)
        hist = returns[max(1, origin_idx - rolling_weeks + 1) : origin_idx + 1]
        hist = hist[np.isfinite(hist)]
        hist_mean = float(np.mean(hist)) if len(hist) else np.nan
        hist_std = float(np.std(hist, ddof=1)) if len(hist) > 1 else np.nan
        best_h = None
        best_z = np.nan
        best_r = np.nan
        for h in range(1, cfg.horizon + 1):
            idx = origin_idx + h
            if idx >= len(returns) or not np.isfinite(hist_std) or hist_std <= 1e-12:
                continue
            z = float((returns[idx] - hist_mean) / hist_std)
            if best_h is None or abs(z) > abs(best_z):
                best_h = h
                best_z = z
                best_r = float(returns[idx])
        is_extreme = bool(best_h is not None and abs(best_z) >= sigma)
        direction = "non_spike"
        if is_extreme and best_z > 0:
            direction = "upward_spike"
        elif is_extreme and best_z < 0:
            direction = "downward_crash"
        rows.append({
            "origin_date": row["origin_date"],
            "origin_idx": origin_idx,
            "target": target,
            "rolling_weeks": rolling_weeks,
            "sigma_threshold": sigma,
            "hist_mean_logret": hist_mean,
            "hist_std_logret": hist_std,
            "max_abs_z_horizon": best_h,
            "max_abs_z_logret": best_r,
            "max_abs_z_score": best_z,
            "is_extreme": is_extreme,
            "regime": "spike" if is_extreme else "non_spike",
            "direction": direction,
        })
    labels = pd.DataFrame(rows)
    origin_extreme = labels.groupby("origin_date")["is_extreme"].sum().reset_index()
    origin_extreme["is_extreme_origin"] = origin_extreme["is_extreme"] >= 1
    return labels.merge(origin_extreme[["origin_date", "is_extreme_origin"]], on="origin_date", how="left")


# ===== Embedded RAG4CTS/run_logging.py =====
import json
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd



def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


def make_run_id(prefix: str = "run") -> str:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    return f"{prefix}-{stamp}"


def _json_default(value: Any) -> str:
    if isinstance(value, Path):
        return str(value)
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return str(value)


class RunLogger:
    """Run-scoped audit logger for experiment-comparison artifacts.

    The logger writes structured CSV/JSON artifacts under
    ``cfg.out_dir / "runs" / run_id``. It does not create zip archives or copy
    the full input dataset.
    """

    def __init__(self, cfg: V210Config, run_id: str | None = None):
        self.cfg = cfg
        self.run_id = run_id or make_run_id()
        self.run_dir = cfg.run_log_root / self.run_id
        self.run_dir.mkdir(parents=True, exist_ok=True)
        self.started_at = utc_now_iso()
        self.artifacts: list[dict[str, Any]] = []
        self.stages: list[dict[str, Any]] = []
        self.errors: list[dict[str, Any]] = []
        self.manifest_path = self.run_dir / "manifest.json"
        self.finalize("running", completed=False)

    def relative_to_run(self, path: str | Path) -> str:
        p = Path(path)
        if not p.is_absolute():
            return str(p)
        try:
            return str(p.relative_to(self.run_dir))
        except ValueError:
            return str(p)

    def _resolve_run_path(self, relative_path: str | Path) -> Path:
        path = Path(relative_path)
        if path.is_absolute():
            raise ValueError("RunLogger paths must be relative to the run directory.")
        if any(part == ".." for part in path.parts):
            raise ValueError("RunLogger paths must not escape the run directory.")
        return self.run_dir / path

    def record_artifact(self, kind: str, path: str | Path, metadata: dict[str, Any] | None = None) -> None:
        p = Path(path)
        record = {
            "kind": kind,
            "path": self.relative_to_run(p),
            "created_at": utc_now_iso(),
        }
        if metadata:
            record["metadata"] = metadata
        self.artifacts.append(record)

    def write_json(
        self,
        relative_path: str | Path,
        payload: object,
        *,
        kind: str,
        metadata: dict[str, Any] | None = None,
    ) -> Path:
        path = self._resolve_run_path(relative_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=_json_default), encoding="utf-8")
        self.record_artifact(kind, path, metadata)
        return path

    def write_frame(
        self,
        relative_path: str | Path,
        df: pd.DataFrame,
        *,
        kind: str,
        metadata: dict[str, Any] | None = None,
    ) -> Path:
        path = self._resolve_run_path(relative_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(path, index=False)
        info = {"rows": int(len(df)), "columns": list(df.columns)}
        if metadata:
            info.update(metadata)
        self.record_artifact(kind, path, info)
        return path

    def record_stage(self, stage: str, status: str, metadata: dict[str, Any] | None = None) -> None:
        record = {"stage": stage, "status": status, "timestamp": utc_now_iso()}
        if metadata:
            record["metadata"] = metadata
        self.stages.append(record)

    def record_error(self, stage: str, exc: BaseException | str, metadata: dict[str, Any] | None = None) -> None:
        if isinstance(exc, BaseException):
            error_type = type(exc).__name__
            message = str(exc)
            tb = "".join(traceback.format_exception(type(exc), exc, exc.__traceback__))
        else:
            error_type = "Error"
            message = str(exc)
            tb = None
        record: dict[str, Any] = {
            "timestamp": utc_now_iso(),
            "stage": stage,
            "error_type": error_type,
            "message": message,
        }
        if tb:
            record["traceback"] = tb
        if metadata:
            record["metadata"] = metadata
        self.errors.append(record)
        errors_path = self.run_dir / "errors.jsonl"
        errors_path.parent.mkdir(parents=True, exist_ok=True)
        with errors_path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(record, ensure_ascii=False, default=_json_default) + "\n")
        if not any(a["path"] == "errors.jsonl" for a in self.artifacts):
            self.record_artifact("errors", errors_path)

    def write_config_runtime(
        self,
        *,
        eval_origins: list[str] | None = None,
        model_switches: dict[str, Any] | None = None,
        runtime: dict[str, Any] | None = None,
        extra: dict[str, Any] | None = None,
    ) -> Path:
        data_path = Path(self.cfg.data_path)
        payload: dict[str, Any] = {
            "run_id": self.run_id,
            "data_path": str(self.cfg.data_path),
            "data_file_size": data_path.stat().st_size if data_path.exists() else None,
            "out_dir": str(self.cfg.out_dir),
            "targets": list(self.cfg.targets),
            "eval_origins": eval_origins or [],
            "horizon": self.cfg.horizon,
            "model_switches": model_switches or {},
            "training_settings": {
                "learning_rate": self.cfg.learning_rate,
                "scheduler_kwargs": self.cfg.scheduler_kwargs,
                "scaler_type": self.cfg.scaler_type,
                "input_size": self.cfg.input_size,
                "batch_size": self.cfg.batch_size,
                "valid_batch_size": self.cfg.valid_batch_size,
                "windows_batch_size": self.cfg.windows_batch_size,
                "inference_windows_batch_size": self.cfg.inference_windows_batch_size,
                "max_steps": self.cfg.max_steps,
                "val_check_steps": self.cfg.val_check_steps,
                "early_stop_patience_steps": self.cfg.early_stop_patience_steps,
                "seed": self.cfg.seed,
            },
            "runtime": runtime or self.cfg.neuralforecast_trainer_kwargs(),
        }
        if extra:
            payload.update(extra)
        return self.write_json("config_runtime.json", payload, kind="config_runtime")

    def run_log_reference(self) -> dict[str, object]:
        return {
            "run_id": self.run_id,
            "root": str(self.run_dir),
            "manifest": str(self.manifest_path),
            "artifact_count": len(self.artifacts),
            "error_count": len(self.errors),
        }

    def manifest(self, status: str, completed: bool = True) -> dict[str, Any]:
        return {
            "run_id": self.run_id,
            "status": status,
            "started_at": self.started_at,
            "completed_at": utc_now_iso() if completed else None,
            "artifacts": self.artifacts,
            "stages": self.stages,
            "errors": self.errors,
        }

    def finalize(self, status: str = "success", *, completed: bool = True) -> Path:
        self.manifest_path.write_text(
            json.dumps(self.manifest(status, completed=completed), indent=2, ensure_ascii=False, default=_json_default),
            encoding="utf-8",
        )
        return self.manifest_path


# ===== Embedded RAG4CTS/pipeline.py =====
import json
from pathlib import Path
from typing import Sequence



def ensure_output_dirs(cfg: V210Config) -> None:
    for path in [cfg.out_dir, cfg.table_dir, cfg.graph_dir]:
        path.mkdir(parents=True, exist_ok=True)


def prepare_inputs(cfg: V210Config, max_origins: int | None = None, targets: Sequence[str] | None = None):
    raw, df, audit = load_selected_weekly_data(cfg.data_path, cfg)
    selected_targets = cfg.targets_to_run(targets)
    returns = target_returns(df, cfg, selected_targets)
    origin_panel, origin_indices, eval_dates = build_origin_panel(df, cfg, selected_targets, max_origins=max_origins)
    return raw, df, audit, returns, origin_panel, origin_indices, eval_dates


def write_core_tables(cfg: V210Config, df: pd.DataFrame, audit: pd.DataFrame, origin_panel: pd.DataFrame) -> None:
    ensure_output_dirs(cfg)
    audit.to_csv(cfg.table_dir / "selected_variable_audit.csv", index=False)
    df.to_csv(cfg.table_dir / "weekly_2011_selected_data.csv", index=False)
    origin_panel.to_csv(cfg.table_dir / "post2023_origin_panel.csv", index=False)
    build_spike_labels(df, origin_panel, cfg).to_csv(cfg.table_dir / "spike_labels_52w_3sigma.csv", index=False)



def write_paper_plots(cfg: V210Config, df: pd.DataFrame, predictions: pd.DataFrame, spike_labels: pd.DataFrame, retrieval_audit: pd.DataFrame | None = None) -> None:
    if predictions.empty:
        return
    import matplotlib.pyplot as plt

    cfg.graph_dir.mkdir(parents=True, exist_ok=True)
    pred = predictions.copy()
    pred["origin_date_dt"] = pd.to_datetime(pred["origin_date"])
    h_eval = int(cfg.horizon)
    for target, tg in pred[pred["horizon"].eq(h_eval)].groupby("target"):
        fig, ax = plt.subplots(figsize=(12, 5))
        actual = tg[["origin_date_dt", "actual_price"]].drop_duplicates().sort_values("origin_date_dt")
        ax.plot(actual["origin_date_dt"], actual["actual_price"], color="black", linewidth=2.2, label="Actual")
        for model, mg in tg.groupby("model"):
            mg = mg.sort_values("origin_date_dt")
            ax.plot(mg["origin_date_dt"], mg["pred_price"], marker="o", markersize=2.5, linewidth=1.2, label=model)
        if not spike_labels.empty:
            lab = spike_labels[(spike_labels["target"].eq(target)) & (spike_labels["is_extreme"].eq(True))]
            for d in pd.to_datetime(lab["origin_date"].unique()):
                ax.axvline(d, color="red", alpha=0.18, linestyle="--", linewidth=1.0)
        ax.set_title(f"{target} h={h_eval} actual vs forecasts")
        ax.set_xlabel("origin date")
        ax.set_ylabel("price")
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8, ncol=2)
        fig.tight_layout()
        fig.savefig(cfg.graph_dir / f"{target}_h{h_eval}_actual_vs_forecasts.png", dpi=180)
        plt.close(fig)

    if retrieval_audit is None or retrieval_audit.empty or "audit_type" not in retrieval_audit.columns:
        return
    donors = retrieval_audit[retrieval_audit["audit_type"].eq("donor_context")].copy()
    if donors.empty:
        return
    if not spike_labels.empty and spike_labels["is_extreme"].any():
        priority = spike_labels[spike_labels["is_extreme"]][["origin_date", "target"]].drop_duplicates()
        donors = donors.merge(priority.assign(_priority=1), on=["origin_date", "target"], how="left")
        donors = donors.sort_values(["_priority", "target", "origin_date"], ascending=[False, True, True])
    first = donors.iloc[0]
    target = str(first["target"])
    origin_date = str(first["origin_date"])
    g = donors[(donors["target"].eq(target)) & (donors["origin_date"].eq(origin_date))].head(5)
    origin_idx = int(first["origin_idx"])
    weeks = int(cfg.proposed_input_weeks)
    fig, ax = plt.subplots(figsize=(10, 5))
    q_start = origin_idx - weeks + 1
    q_idx = list(range(q_start, origin_idx + cfg.horizon + 1))
    q_vals = df.loc[q_idx, target].astype(float).to_numpy()
    q_vals = q_vals / q_vals[0] * 100.0
    ax.plot(range(-weeks + 1, cfg.horizon + 1), q_vals, color="black", linewidth=2.4, label="Query/actual path")
    for rank, rec in enumerate(g.itertuples(index=False), start=1):
        ci = int(getattr(rec, "candidate_origin_idx"))
        c_start = ci - weeks + 1
        c_idx = list(range(c_start, ci + cfg.horizon + 1))
        c_vals = df.loc[c_idx, target].astype(float).to_numpy()
        c_vals = c_vals / c_vals[0] * 100.0
        ax.plot(range(-weeks + 1, cfg.horizon + 1), c_vals, linewidth=1.4, alpha=0.75, label=f"Donor {rank}: {getattr(rec, 'candidate_origin_date')}")
    ax.axvline(0, color="gray", linestyle="--", linewidth=1.0)
    ax.set_title(f"Retrieved donor paths: {target} origin={origin_date}")
    ax.set_xlabel("relative week from origin")
    ax.set_ylabel("indexed price (window start=100)")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(cfg.graph_dir / f"donor_path_{target}_{origin_date}.png", dpi=180)
    plt.close(fig)


def build_manifest(cfg: V210Config, eval_origin_dates: list[str], targets_to_run: Sequence[str], run_neuralforecast: bool, run_chronos: bool, run_proposed: bool) -> dict[str, object]:
    return {
        "version": "211-paper-aligned-h2-fold48",
        "method": "v210_neuralforecast_chronos_baseline_tscv_post2023",
        "scope": cfg.panel_name,
        "input_data": str(cfg.data_path),
        "origin_start": str(cfg.origin_start.date()),
        "eval_origins": eval_origin_dates,
        "targets": list(targets_to_run),
        "horizon": cfg.horizon,
        "val_size": cfg.val_size,
        "test_size": cfg.test_size,
        "step_size": cfg.step_size,
        "common_training_settings": {
            "loss": "MSE", "valid_loss": "MSE", "optimizer": "AdamW", "learning_rate": cfg.learning_rate,
            "scheduler": "ReduceLROnPlateau", "scheduler_kwargs": cfg.scheduler_kwargs, "scaler": cfg.scaler_type,
            "input_size": cfg.input_size, "batch_size": cfg.batch_size, "valid_batch_size": cfg.valid_batch_size,
            "windows_batch_size": cfg.windows_batch_size, "inference_windows_batch_size": cfg.inference_windows_batch_size,
            "max_steps": cfg.max_steps, "val_check_steps": cfg.val_check_steps,
            "early_stop_patience_steps": cfg.early_stop_patience_steps, "seed": cfg.seed,
        },
        "evaluation": {
            "primary_metric": "MAPE",
            "reported_metrics": ["MSE", "RMSE", "MAE", "MAPE", "sMAPE"],
            "metric_scale": "price level actual_price vs pred_price",
        },
        "runtime": {
            "require_gpu_for_full_run": cfg.require_gpu_for_full_run,
            "neuralforecast": cfg.neuralforecast_trainer_kwargs(),
            "chronos_device_map": cfg.chronos_device_map,
            "chronos_dtype": cfg.chronos_dtype,
        },
        "models": {
            "neuralforecast": ["TimeXer", "iTransformer", "GRU"] if run_neuralforecast else [],
            "chronos2_query_only": run_chronos,
            "proposed": "Proposed_DynamicK_RetrievedContext_Chronos2" if run_proposed else None,
        },
        "data_policy": {
            "weekly_only": True, "daily_data_used": False, "data_added": False,
            "fixed_start": str(cfg.fixed_start.date()), "drop_variables_missing_at_fixed_start": True,
            "no_leading_zero_fill": True,
        },
        "constraints": {
            "requested_actuals_used_for_training_selection_or_reranking": False,
            "requested_actuals_used_for_metrics_plots_diagnostics_only": True,
            "event_gates_spike_labels": False,
            "spike_definition": "diagnostic only: previous 52-week rolling log-return mean +/- 3 std",
            "qtail_floor_max_correction": False,
            "origin_specific_manual_rule": False,
            "analog_fallback": False,
        },
    }


def _manifest_outputs(cfg: V210Config) -> list[str]:
    if not cfg.out_dir.exists():
        return []
    outputs: list[str] = []
    for path in cfg.out_dir.rglob("*"):
        if not path.is_file():
            continue
        rel = path.relative_to(cfg.out_dir)
        if rel.parts and rel.parts[0] == "runs":
            continue
        outputs.append(str(rel))
    return sorted(outputs)


def write_manifest(cfg: V210Config, manifest: dict[str, object], run_log: dict[str, object] | None = None) -> None:
    ensure_output_dirs(cfg)
    manifest = dict(manifest)
    outputs = _manifest_outputs(cfg)
    manifest_rel = str((cfg.table_dir / "manifest.json").relative_to(cfg.out_dir))
    if manifest_rel not in outputs:
        outputs.append(manifest_rel)
    manifest["outputs"] = sorted(outputs)
    if run_log is not None:
        manifest["run_log"] = run_log
    (cfg.table_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")


# ===== Embedded scripts/run_main_pipeline.py behavior without local imports =====

def parse_targets(value: str | None) -> list[str] | None:
    if value is None or value.strip().lower() in {"", "all"}:
        return None
    return [x.strip() for x in value.split(",") if x.strip()]


def parse_models(value: str) -> set[str]:
    aliases = {
        "all": {"neuralforecast", "chronos", "proposed"},
        "nf": {"neuralforecast"},
        "neuralforecast": {"neuralforecast"},
        "chronos": {"chronos"},
        "chronos-query": {"chronos"},
        "proposed": {"proposed"},
        "prep": set(),
        "prep-only": set(),
    }
    out: set[str] = set()
    for part in [x.strip().lower() for x in value.split(",") if x.strip()]:
        if part not in aliases:
            raise ValueError(f"Unknown model selector {part!r}; use all,nf,chronos,proposed,prep")
        out.update(aliases[part])
    return out


def run_pipeline_inline(
    *,
    data: str | Path = "data/df_0508.csv",
    out_dir: str | Path = "v211_paper_h2_fold48_outputs",
    origin_start: str = "2023-01-01",
    fixed_start: str | None = "2011-01-10",
    horizon: int = 2,
    step_size: int = 1,
    panel_name: str | None = None,
    targets: str | None = None,
    models: str = "all",
    max_origins: int | None = None,
    no_gpu_required: bool = False,
) -> int:
    model_set = parse_models(models)
    target_list = parse_targets(targets)
    panel = panel_name or f"paper_post2023_h{horizon}_fold48_step{step_size}_all_models"

    cfg = default_config().with_paths(data_path=data, out_dir=out_dir)
    cfg = replace(
        cfg,
        fixed_start=pd.Timestamp(fixed_start) if fixed_start else cfg.fixed_start,
        origin_start=pd.Timestamp(origin_start),
        horizon=horizon,
        test_size=horizon,
        step_size=step_size,
        panel_name=panel,
    ).with_local_gpu_runtime(require_gpu=not no_gpu_required)

    selected_targets = cfg.targets_to_run(target_list)
    run_logger = RunLogger(cfg)
    predictions: list[pd.DataFrame] = []

    try:
        raw, df, audit, returns, origin_panel, _origin_indices, eval_dates = prepare_inputs(
            cfg,
            max_origins=max_origins,
            targets=selected_targets,
        )
        write_core_tables(cfg, df, audit, origin_panel)
        run_logger.record_stage(
            "prepare_inputs",
            "success",
            {
                "raw_shape": raw.shape,
                "selected_shape": df.shape,
                "origin_panel_shape": origin_panel.shape,
                "first_origin": eval_dates[0],
                "last_origin": eval_dates[-1],
                "origin_count": len(eval_dates),
            },
        )
        run_logger.write_config_runtime(
            eval_origins=eval_dates,
            model_switches={
                "neuralforecast": "neuralforecast" in model_set,
                "chronos2_query_only": "chronos" in model_set,
                "proposed": "proposed" in model_set,
            },
            extra={"max_origins": max_origins, "selected_targets": list(selected_targets)},
        )

        if "neuralforecast" in model_set:
            nf_pred, nf_raw = run_neuralforecast_tscv(df, returns, cfg, eval_dates, selected_targets)
            predictions.append(nf_pred)
            run_logger.write_frame("predictions_neuralforecast.csv", nf_pred, kind="predictions")
            run_logger.write_frame("raw_neuralforecast_cv.csv", nf_raw, kind="raw_predictions")
            run_logger.record_stage("neuralforecast", "success", {"rows": len(nf_pred)})

        if "chronos" in model_set:
            chronos_pred = run_chronos_query_tscv(df, returns, cfg, origin_panel, selected_targets)
            predictions.append(chronos_pred)
            run_logger.write_frame("predictions_chronos_query.csv", chronos_pred, kind="predictions")
            run_logger.record_stage("chronos_query", "success", {"rows": len(chronos_pred)})

        if "proposed" in model_set:
            proposed_pred, retrieval_audit = run_proposed_tscv(df, returns, cfg, origin_panel, selected_targets)
            predictions.append(proposed_pred)
            run_logger.write_frame("predictions_proposed.csv", proposed_pred, kind="predictions")
            run_logger.write_frame("retrieval_audit.csv", retrieval_audit, kind="retrieval_audit")
            run_logger.record_stage("proposed", "success", {"rows": len(proposed_pred), "audit_rows": len(retrieval_audit)})

        all_predictions = pd.concat(predictions, ignore_index=True) if predictions else pd.DataFrame()
        spike_labels = build_spike_labels(df, origin_panel, cfg)
        retrieval_audit_frame = locals().get("retrieval_audit", pd.DataFrame())
        if not all_predictions.empty:
            all_predictions = all_predictions.merge(
                spike_labels[["origin_date", "target", "is_extreme", "regime", "direction", "max_abs_z_score", "max_abs_z_horizon"]],
                on=["origin_date", "target"],
                how="left",
            )
            run_logger.write_frame("predictions.csv", all_predictions, kind="predictions")
            run_logger.write_frame("metrics_summary.csv", summarize_predictions(all_predictions, cfg), kind="metrics_summary")
            run_logger.write_frame("metrics_by_regime.csv", summarize_predictions_by_regime(all_predictions, cfg), kind="metrics_by_regime")
            run_logger.write_frame("spike_labels_52w_3sigma.csv", spike_labels, kind="spike_labels")
            write_paper_plots(cfg, df, all_predictions, spike_labels, retrieval_audit_frame)

        manifest = build_manifest(
            cfg,
            eval_dates,
            selected_targets,
            "neuralforecast" in model_set,
            "chronos" in model_set,
            "proposed" in model_set,
        )
        run_logger.record_stage("manifest", "success")
        run_logger.finalize("success")
        write_manifest(cfg, manifest, run_log=run_logger.run_log_reference())
        print("status=success")
        print(f"origins={len(eval_dates)} first={eval_dates[0]} last={eval_dates[-1]}")
        print(f"out_dir={cfg.out_dir}")
        print(f"run_dir={run_logger.run_dir}")
        print(f"manifest={cfg.table_dir / 'manifest.json'}")
        return 0
    except BaseException as exc:
        run_logger.record_error("pipeline", exc)
        run_logger.finalize("failed")
        raise


In [ ]:
# 3. Exact run settings matching the requested CLI.
from pathlib import Path

DATA_PATH = Path("data/df_0508.csv")
OUT_DIR = Path("v211_paper_h2_fold48_outputs")

FIXED_START = "2011-01-10"
ORIGIN_START = "2023-01-01"
HORIZON = 2
STEP_SIZE = 1
MODELS = "all"
PANEL_NAME = None
TARGETS = None
MAX_ORIGINS = 48
NO_GPU_REQUIRED = False  # Same as omitting --no-gpu-required.

print("Equivalent CLI:")
print(
    "python scripts/run_main_pipeline.py "
    "--data data/df_0508.csv "
    f"--fixed-start {FIXED_START} "
    f"--origin-start {ORIGIN_START} "
    f"--horizon {HORIZON} "
    f"--step-size {STEP_SIZE} "
    f"--models {MODELS} "
    f"--max-origins {MAX_ORIGINS} "
    "--out-dir v211_paper_h2_fold48_outputs"
)
print("Resolved models:", parse_models(MODELS))


In [ ]:
# 4. Pre-run checks. This does not train or forecast.
if not DATA_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {DATA_PATH}. Run the upload cell first.")

print("CSV:", DATA_PATH, f"({DATA_PATH.stat().st_size:,} bytes)")
print("Output directory:", OUT_DIR)
print("Runtime GPU summary:", torch_gpu_summary())

if NO_GPU_REQUIRED is False:
    gpu = torch_gpu_summary()
    if not gpu.get("cuda_available"):
        raise RuntimeError("GPU/CUDA is required for this all-model run. In Colab, set Runtime > Change runtime type > GPU.")


In [ ]:
# 5. Run the full pipeline. This is the only cell that starts model execution.
run_pipeline_inline(
    data=DATA_PATH,
    out_dir=OUT_DIR,
    fixed_start=FIXED_START,
    origin_start=ORIGIN_START,
    horizon=HORIZON,
    step_size=STEP_SIZE,
    models=MODELS,
    panel_name=PANEL_NAME,
    targets=TARGETS,
    max_origins=MAX_ORIGINS,
    no_gpu_required=NO_GPU_REQUIRED,
)
